In [8]:
# scripts/make_eda.py
"""
Excel → raw CSV → light cleaning → clean CSV + long-form EDA
Repo-relative paths so it works right after cloning.

Usage
-----
# default: use 'data/raw/CA23107participant list.xlsx' if present,
# otherwise the first .xlsx/.xls in data/raw/
python scripts/make_eda.py

# or specify a file (absolute or relative to repo root) and sheet
python scripts/make_eda.py --input data/raw/participants.xlsx --sheet 0
"""

from __future__ import annotations
from pathlib import Path
from typing import Iterable, Optional
import argparse
import re
import sys

import numpy as np
import pandas as pd


# --------------------------- repo root detection --------------------------- #
def _find_repo_root() -> Path:
    """Work in scripts, terminal, or notebooks (no __file__)."""
    try:
        here = Path(__file__).resolve()
        return here.parent.parent  # …/scripts → repo root
    except NameError:
        cwd = Path.cwd().resolve()
        if (cwd / "data").is_dir() and (cwd / "scripts").is_dir():
            return cwd
        if cwd.name == "scripts" and (cwd.parent / "data").is_dir():
            return cwd.parent
        cur = cwd
        for _ in range(5):
            if (cur / ".git").is_dir() or ((cur / "data").is_dir() and (cur / "scripts").is_dir()):
                return cur
            cur = cur.parent
        return cwd


ROOT = _find_repo_root()
RAW_DIR = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
OUT_DIR  = ROOT / "outputs"
PROC_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root →", ROOT)


# ------------------------------- helpers ---------------------------------- #
def write_csv(df: pd.DataFrame, path: Path) -> Path:
    """Save a DataFrame to UTF-8 CSV (with BOM for Excel friendliness)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Saved → {path.relative_to(ROOT)}")
    return path


def snake_case(name: str) -> str:
    """Normalize headers to lowercase_with_underscores."""
    name = str(name).replace("\u00A0", " ")
    name = re.sub(r"[^\w\s\-]+", " ", name)
    name = " ".join(name.split()).strip().lower().replace("-", " ")
    return re.sub(r"\s+", "_", name)


def clean_text_series(s: pd.Series) -> pd.Series:
    """Gentle text clean for object columns."""
    out = (
        s.astype(str)
         .str.replace(r"\*", "", regex=True)
         .str.replace("\u00A0", " ", regex=False)
         .str.replace(r"\s+", " ", regex=True)
         .str.strip()
    )
    return out.replace({"": np.nan, "nan": np.nan})


_YN_PATTERNS = [re.compile(p, re.I) for p in (
    r"^wg\s*\d+",
    r"^is_",
    r"\bmc\b|\bmc_member\b",
    r"\bcore\b|\bcore_group\b",
    r"^wg_member$",
)]


def looks_like_yn(names: Iterable[str]) -> list[bool]:
    return [any(p.search(str(c).lower()) for p in _YN_PATTERNS) for c in names]


def normalize_yes_no(s: pd.Series, pending_policy: str = "false") -> pd.Series:
    """Map common yes/no tokens → True/False. Handles 'pending' per policy."""
    true_tokens  = {"y", "yes", "true", "1", "member"}
    false_tokens = {"n", "no", "false", "0"}

    raw = s.astype(str).str.strip().str.lower()
    is_pending = raw.str.contains(r"\bpending\b", na=False)
    is_yesish  = raw.str.contains(r"^\s*yes\b", na=False)

    out = pd.Series(index=s.index, dtype="object")
    out[(raw.isin(true_tokens) | is_yesish) & ~is_pending] = True
    out[raw.isin(false_tokens)] = False

    if pending_policy == "false":
        out[is_pending] = False
    elif pending_policy == "na":
        out[is_pending] = pd.NA

    return out.where(out.notna(), s)


def gentle_type_infer(s: pd.Series) -> pd.Series:
    """Two-pass, conservative typing: numeric then datetime."""
    if s.dtype == "object":
        raw = s.astype(str).str.replace(",", "").str.strip()
        looks_num = raw.str.match(r"^-?\d+(\.\d+)?$", na=False)
        if looks_num.mean() >= 0.6:
            return pd.to_numeric(raw, errors="coerce")

    if s.dtype == "object" or s.dtype.kind in "Mm":
        if s.dtype.kind in "Mm":
            return s
        sample = s.astype(str).str.lower()
        # FIXED: call .str.contains(...) (no stray assignment)
        looks_date = sample.str.contains(
            r"\d{1,4}[-/]\d{1,2}[-/]\d{1,4}|jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec",
            regex=True, na=False
        )
        if looks_date.mean() >= 0.5:
            parsed = pd.to_datetime(s, errors="coerce", dayfirst=True)
            if parsed.notna().mean() >= 0.5:
                return parsed
    return s


def clean_country_column(df: pd.DataFrame) -> pd.DataFrame:
    """Add 'country_clean' if a 'country' column exists."""
    matches = [c for c in df.columns if c.lower() == "country"]
    if not matches:
        return df
    col = matches[0]
    df["country_clean"] = (
        df[col].astype(str)
              .str.replace(r"\(.*?\)", "", regex=True)
    )
    df["country_clean"] = clean_text_series(df["country_clean"])

    aliases = {
        "uk": "United Kingdom", "united kingdom": "United Kingdom", "great britain": "United Kingdom",
        "czech republic": "Czechia",
        "turkey": "Türkiye", "turkiye": "Türkiye",
        "north macedonia": "North Macedonia", "macedonia": "North Macedonia",
        "bosnia and herzegovina": "Bosnia and Herzegovina",
        "moldova": "Moldova", "russia": "Russia",
        "ivory coast": "Côte d'Ivoire", "cote d ivoire": "Côte d'Ivoire",
        "republic of kosovo": "Kosovo", "kosovo": "Kosovo",
    }
    low = df["country_clean"].str.lower()
    df.loc[low.isin(aliases), "country_clean"] = low.map(aliases)
    return df


def value_counts_preview(s: pd.Series, n: int = 10) -> str:
    vc = s.astype("object").fillna("<NA>").value_counts(dropna=False).head(n)
    return "; ".join(f"{k}: {int(v)}" for k, v in vc.items())


def mark_section(df: pd.DataFrame, name: str) -> pd.DataFrame:
    out = df.copy()
    out.insert(0, "section", name)
    return out


# -------------------------- CLI / input selection -------------------------- #
def find_default_excel(raw_dir: Path) -> Optional[Path]:
    """Prefer your CA23107 file; else first .xlsx/.xls."""
    preferred = raw_dir / "CA23107participant list.xlsx"
    if preferred.exists():
        return preferred
    files = sorted(list(raw_dir.glob("*.xlsx")) + list(raw_dir.glob("*.xls")))
    return files[0] if files else None


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description="Build data_raw.csv, data_clean.csv, eda_overview.csv")
    p.add_argument("--input", "-i", type=str, default=None,
                   help="Excel file to process (default: CA23107… or first .xlsx in data/raw/)")
    p.add_argument("--sheet", "-s", default=0,
                   help="Sheet index (0) or name (string). Default: 0")
    p.add_argument("--topn", type=int, default=25,
                   help="Top N categories captured in EDA 'top_values' (default 25)")

    # Jupyter/IDE inject unknown args (e.g., --f=...). Ignore them safely.
    if "ipykernel" in sys.modules or "IPython" in sys.modules:
        args, _unknown = p.parse_known_args()
    else:
        args = p.parse_args()
    return args


# -------------------------------- pipeline --------------------------------- #
def main() -> None:
    args = parse_args()

    # Resolve input Excel path (repo-relative if not absolute)
    if args.input:
        excel_path = Path(args.input)
        if not excel_path.is_absolute():
            excel_path = (ROOT / excel_path).resolve()
    else:
        excel_path = find_default_excel(RAW_DIR) or Path()

    if not excel_path.exists():
        print("ERROR: Excel not found.")
        print(f"- Looked for: {RAW_DIR / 'CA23107participant list.xlsx'}")
        print(f"- Or first .xlsx/.xls in: {RAW_DIR}")
        print(f"- Or pass a file:  python scripts/make_eda.py --input data/raw/yourfile.xlsx")
        sys.exit(1)

    sheet = args.sheet
    top_n = args.topn

    # Friendly path print
    rel = excel_path
    try:
        rel = excel_path.relative_to(ROOT)
    except Exception:
        pass
    print(f"Reading Excel: {rel}")

    # 1) Read Excel as-is
    df_raw = pd.read_excel(excel_path, sheet_name=sheet)

    # 2) Save verbatim copy → data/raw/data_raw.csv
    raw_csv = RAW_DIR / "data_raw.csv"
    write_csv(df_raw, raw_csv)

    # 3) Detect Y/N-like columns BEFORE cleaning (intent by original names)
    yn_flags_before = looks_like_yn(df_raw.columns)
    yn_candidates_raw = [c for c, is_yn in zip(df_raw.columns, yn_flags_before) if is_yn]
    yn_before_df = pd.DataFrame({
        "original_name": yn_candidates_raw,
        "normalized_name_if_any": [snake_case(c) for c in yn_candidates_raw],
        "dtype_raw": [str(df_raw[c].dtype) for c in yn_candidates_raw],
        "top_values_preview": [value_counts_preview(df_raw[c], n=10) for c in yn_candidates_raw],
    })

    # 4) Light cleaning
    df = df_raw.copy()

    # 4a) normalize headers
    old_to_new = {c: snake_case(c) for c in df.columns}
    df.rename(columns=old_to_new, inplace=True)

    # 4b) clean text columns
    obj_cols = df.select_dtypes(include="object").columns
    if len(obj_cols):
        df[obj_cols] = df[obj_cols].apply(clean_text_series)

    # 4c) normalize likely Y/N columns (by name)
    yn_flags_after = looks_like_yn(df.columns)
    yn_after_names = [c for c, is_yn in zip(df.columns, yn_flags_after) if is_yn]
    for c in yn_after_names:
        df[c] = normalize_yes_no(df[c], pending_policy="false")

    # 4d) conservative type inference
    for c in df.columns:
        df[c] = gentle_type_infer(df[c])

    # 4e) country helper (adds 'country_clean' if 'country' exists)
    df = clean_country_column(df)

    # 5) Save cleaned CSV → data/processed/data_clean.csv
    clean_csv = PROC_DIR / "data_clean.csv"
    write_csv(df, clean_csv)

    # 6) Build a single long-form EDA CSV
    eda_parts: list[pd.DataFrame] = []

    # 6a) dtype changes (before vs after)
    dtype_changes = pd.DataFrame({
        "original_name": list(df_raw.columns),
        "cleaned_name": [old_to_new.get(c, c) for c in df_raw.columns],
        "dtype_raw": [str(df_raw[c].dtype) for c in df_raw.columns],
        "dtype_final": [str(df[old_to_new.get(c, c)].dtype) if old_to_new.get(c, c) in df.columns else "<missing>" for c in df_raw.columns],
    })
    dtype_changes["changed"] = dtype_changes["dtype_raw"] != dtype_changes["dtype_final"]
    eda_parts.append(mark_section(dtype_changes, "dtype_changes"))

    # 6b) Y/N detection before cleaning
    if not yn_before_df.empty:
        eda_parts.append(mark_section(yn_before_df, "yn_detection_before"))

    # 6c) Y/N inference after cleaning
    if yn_candidates_raw:
        raw_to_clean = {raw: old_to_new.get(raw, raw) for raw in yn_candidates_raw}
        rows = []
        for raw, cleaned in raw_to_clean.items():
            if cleaned in df.columns:
                s = df[cleaned]
                looks_bool = (s.dtype == bool) or s.dropna().isin([True, False]).all()
                rows.append({
                    "original_name": raw,
                    "cleaned_name": cleaned,
                    "final_dtype": str(s.dtype),
                    "true_count": int((s == True).sum()) if looks_bool else None,    # noqa: E712
                    "false_count": int((s == False).sum()) if looks_bool else None,  # noqa: E712
                    "na_count": int(s.isna().sum()),
                    "top_values_after": value_counts_preview(s, n=10),
                })
            else:
                rows.append({
                    "original_name": raw, "cleaned_name": cleaned, "final_dtype": "<missing>",
                    "true_count": None, "false_count": None, "na_count": None, "top_values_after": "",
                })
        eda_parts.append(mark_section(pd.DataFrame(rows), "yn_inference_after"))

    # 6d) columns summary
    cols_summary = pd.DataFrame({
        "column": df.columns,
        "dtype": [str(df[c].dtype) for c in df.columns],
        "non_null": [int(df[c].notna().sum()) for c in df.columns],
        "missing": [int(df[c].isna().sum()) for c in df.columns],
        "unique": [int(df[c].nunique(dropna=True)) for c in df.columns],
        "example": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    }).sort_values("column")
    eda_parts.append(mark_section(cols_summary, "columns_summary"))

    # 6e) numeric summary (long format)
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols):
        desc_long = (
            df[num_cols].describe(include="all")
                        .T.reset_index().rename(columns={"index": "column"})
                        .melt(id_vars="column", var_name="metric", value_name="value")
        )
        eda_parts.append(mark_section(desc_long, "numeric_summary"))

    # 6f) top values for object/bool/category columns
    cat_cols = df.select_dtypes(include=["object", "bool", "category"]).columns
    if len(cat_cols):
        rows = []
        for c in cat_cols:
            vc = df[c].astype("object").fillna("<NA>").value_counts(dropna=False).head(top_n)
            rows.extend({"column": c, "value": k, "count": int(v)} for k, v in vc.items())
        eda_parts.append(mark_section(pd.DataFrame(rows), "top_values"))

    # 6g) country counts (if present)
    country_col = [c for c in df.columns if c == "country_clean"] or [c for c in df.columns if c == "country"]
    if country_col:
        country_counts = (
            df[country_col[0]].astype("object").fillna("<NA>")
              .value_counts(dropna=False).rename_axis("country")
              .reset_index(name="count")
        )
        eda_parts.append(mark_section(country_counts, "countries_counts"))

    # 6h) write EDA report → outputs/eda_overview.csv
    eda_overview = pd.concat(eda_parts, ignore_index=True, sort=False) if eda_parts else pd.DataFrame({"section":[]})
    write_csv(eda_overview, OUT_DIR / "eda_overview.csv")

    print("\nAll files written under:")
    print(" -", RAW_DIR.relative_to(ROOT))   # data_raw.csv
    print(" -", PROC_DIR.relative_to(ROOT))  # data_clean.csv
    print(" -", OUT_DIR.relative_to(ROOT))   # eda_overview.csv


if __name__ == "__main__":
    main()


Repo root → C:\Users\James\Documents\GitHub\evidence-map-agrifood
Reading Excel: data\raw\CA23107participant list.xlsx
Saved → data\raw\data_raw.csv
Saved → data\processed\data_clean.csv
Saved → outputs\eda_overview.csv

All files written under:
 - data\raw
 - data\processed
 - outputs


C:\Users\James\AppData\Local\Temp\ipykernel_52588\1911241610.py:171: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  vc = s.astype("object").fillna("<NA>").value_counts(dropna=False).head(n)
C:\Users\James\AppData\Local\Temp\ipykernel_52588\1911241610.py:171: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  vc = s.astype("object").fillna("<NA>").value_counts(dropna=False).head(n)
C:\Users\James\AppData\Local\Temp\ipykernel_52588\1911241610.py:171: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a futu